## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

print('All libraries imported successfully!')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

All libraries imported successfully!
Pandas version: 2.2.2
NumPy version: 2.0.2


## Step 2: Load the Dataset

In [ ]:
# Load main crop production dataset
df = pd.read_csv('crop_dataset.csv')

print('Dataset Shape:', df.shape)
print('\nColumn Names:')
print(df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

Dataset Shape: (132248, 10)

Column Names:
['State', 'District ', 'Crop', 'Crop_Year', 'Season', 'Area ', 'Production', 'Yield', 'Rainfall', 'Temperature']

First 5 rows:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Rainfall,Temperature
0,Andaman and Nicobar Island,NICOBARS,Arecanut,2007.0,Kharif,2439.6,3415.0,1.40,1099.342831,20.792735
1,Andaman and Nicobar Island,NICOBARS,Arecanut,2007.0,Rabi,1626.4,2277.0,1.40,118.911430,27.593758
2,Andaman and Nicobar Island,NICOBARS,Arecanut,2008.0,Autumn,4147.0,3060.0,0.74,243.187611,27.269060
3,Andaman and Nicobar Island,NICOBARS,Arecanut,2008.0,Summer,4147.0,2660.0,0.64,113.934585,32.103427
4,Andaman and Nicobar Island,NICOBARS,Arecanut,2009.0,Autumn,4153.0,3120.0,0.75,145.526618,33.066958


## Step 3: Data Understanding

In [ ]:
# Basic information about the dataset
print('=== DATA TYPES ===')
print(df.dtypes)

print('\n=== DATASET INFO ===')
df.info()

=== DATA TYPES ===
State           object
District        object
Crop            object
Crop_Year      float64
Season          object
Area           float64
Production     float64
Yield          float64
Rainfall       float64
Temperature    float64
dtype: object

=== DATASET INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132248 entries, 0 to 132247
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   State        132248 non-null  object 
 1   District     132247 non-null  object 
 2   Crop         132247 non-null  object 
 3   Crop_Year    132247 non-null  float64
 4   Season       132247 non-null  object 
 5   Area         132247 non-null  float64
 6   Production   129817 non-null  float64
 7   Yield        132247 non-null  float64
 8   Rainfall     132247 non-null  float64
 9   Temperature  132247 non-null  float64
dtypes: float64(6), object(4)
memory usage: 10.1+ MB


In [ ]:
# Statistical summary of numerical columns
print('=== STATISTICAL SUMMARY ===')
df.describe()

=== STATISTICAL SUMMARY ===


,Crop_Year,Area,Production,Yield,Rainfall,Temperature
count,132247.000000,132247.000000,1.298170e+05,132247.000000,132247.000000,132247.000000
mean,2008.866696,8987.258643,7.632429e+05,95.106923,617.489702,24.797162
std,6.508024,30735.087385,1.851428e+07,950.835452,435.099234,5.491641
min,1997.000000,0.100000,0.000000e+00,0.000000,0.000000,3.489216
25%,2003.000000,78.000000,9.100000e+01,0.540000,159.345760,20.944959
50%,2009.000000,500.000000,6.860000e+02,1.000000,698.907503,25.367605
75%,2015.000000,3272.000000,6.345000e+03,2.460000,998.754952,28.746273
max,2019.000000,877029.000000,1.452725e+09,43958.330000,2001.676931,50.714569


In [ ]:
# Check unique values in categorical columns
print('Unique States:', df['State'].nunique())
print('\nStates list:')
print(df['State'].unique())

print('\nUnique Seasons:', df['Season'].nunique())
print('Seasons:', df['Season'].unique())

print('\nUnique Crops:', df['Crop'].nunique())
print('\nYear Range:', df['Crop_Year'].min(), 'to', df['Crop_Year'].max())

Unique States: 18

States list:
['Andaman and Nicobar Island' 'Andhra Pradesh' 'Arunachal Pradesh' 'Assam'
 'Bihar' 'CHANDIGARH' 'Chhattisgarh' 'Dadra and Nagar Haveli'
 'Daman and Diu' 'Delhi' 'Goa' 'Gujarat' 'Haryana' 'Himachal Pradesh'
 'Jammu and Kashmir' 'Jharkhand' 'Karnataka' 'Karnata']

Unique Seasons: 6
Seasons: ['Kharif' 'Rabi' 'Autumn' 'Summer' 'Whole Year' 'Winter' nan]

Unique Crops: 54

Year Range: 1997.0 to 2019.0


## Step 4: Handle Missing Values

In [ ]:
# Check missing values
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

=== MISSING VALUES ===
             Missing Count  Missing %
District                 1   0.000756
Crop                     1   0.000756
Crop_Year                1   0.000756
Season                   1   0.000756
Area                     1   0.000756
Production            2431   1.838213
Yield                    1   0.000756
Rainfall                 1   0.000756
Temperature              1   0.000756


In [ ]:
# Drop rows where Production is missing (our target variable)
print(f'Shape before dropping missing Production: {df.shape}')
df = df.dropna(subset=['Production'])
print(f'Shape after dropping missing Production: {df.shape}')

# Clean column names by stripping whitespace
df.columns = df.columns.str.strip()

# Fill missing Area with median grouped by crop
df['Area'] = df.groupby('Crop')['Area'].transform(lambda x: x.fillna(x.median()))

# Drop remaining rows with missing values
df = df.dropna()
print(f'Final shape after all cleaning: {df.shape}')

Shape before dropping missing Production: (129817, 10)
Shape after dropping missing Production: (129817, 10)
Final shape after all cleaning: (129817, 10)


## Step 5: Remove Duplicates

In [ ]:
print(f'Duplicate rows before: {df.duplicated().sum()}')
df = df.drop_duplicates()
print(f'Duplicate rows after: {df.duplicated().sum()}')

Duplicate rows before: 0
Duplicate rows after: 0


## Step 6: Feature Engineering

In [ ]:
# Create the target variable: Yield = Production / Area (tonnes per hectare)
df['Yield'] = df['Production'] / df['Area']

# Remove extreme outliers in Yield using IQR method
Q1 = df['Yield'].quantile(0.25)
Q3 = df['Yield'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f'Yield range before outlier removal: {df["Yield"].min():.2f} to {df["Yield"].max():.2f}')
df = df[(df['Yield'] >= lower) & (df['Yield'] <= upper)]
print(f'Yield range after outlier removal: {df["Yield"].min():.2f} to {df["Yield"].max():.2f}')
print(f'Shape after outlier removal: {df.shape}')

Yield range before outlier removal: 0.00 to 43958.33
Yield range after outlier removal: 0.00 to 5.39
Shape after outlier removal: (110978, 10)


In [ ]:
# Clean Season column (strip whitespace)
df['Season'] = df['Season'].str.strip()

# Encode Season as numeric
season_map = {
    'Kharif': 0,
    'Rabi': 1,
    'Whole Year': 2,
    'Summer': 3,
    'Winter': 4,
    'Autumn': 5
}
df['Season_Encoded'] = df['Season'].map(season_map)
df['Season_Encoded'] = df['Season_Encoded'].fillna(-1).astype(int)

# Label encode State and Crop
le_state = LabelEncoder()
le_crop = LabelEncoder()
df['State_Encoded'] = le_state.fit_transform(df['State'])
df['Crop_Encoded'] = le_crop.fit_transform(df['Crop'])

print('Encoding done!')
print(df[['State', 'State_Encoded', 'Crop', 'Crop_Encoded', 'Season', 'Season_Encoded']].head(10))

Encoding done!
                        State  State_Encoded      Crop  Crop_Encoded  \
0  Andaman and Nicobar Island              0  Arecanut             0   
1  Andaman and Nicobar Island              0  Arecanut             0   
2  Andaman and Nicobar Island              0  Arecanut             0   
3  Andaman and Nicobar Island              0  Arecanut             0   
4  Andaman and Nicobar Island              0  Arecanut             0   
5  Andaman and Nicobar Island              0  Arecanut             0   
6  Andaman and Nicobar Island              0  Arecanut             0   
7  Andaman and Nicobar Island              0  Arecanut             0   
8  Andaman and Nicobar Island              0  Arecanut             0   
9  Andaman and Nicobar Island              0  Arecanut             0   

       Season  Season_Encoded  
0      Kharif               0  
1        Rabi               1  
2      Autumn               5  
3      Summer               3  
4      Autumn               5  


In [ ]:
# Add a log-transformed yield column (to handle skewness)
df['Log_Yield'] = np.log1p(df['Yield'])
df['Log_Area'] = np.log1p(df['Area'])
df['Log_Production'] = np.log1p(df['Production'])

print('Feature engineering complete. New columns added:')
print(df.columns.tolist())

Feature engineering complete. New columns added:
['State', 'District', 'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Yield', 'Rainfall', 'Temperature', 'Season_Encoded', 'State_Encoded', 'Crop_Encoded', 'Log_Yield', 'Log_Area', 'Log_Production']


## Step 7: Save Cleaned Dataset

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/crop_cleaned.csv', index=False)
print(f'Cleaned dataset saved to data/processed/crop_cleaned.csv')
print(f'Final shape: {df.shape}')
print('\nFinal columns:')
print(df.columns.tolist())

Cleaned dataset saved to data/processed/crop_cleaned.csv
Final shape: (110978, 16)

Final columns:
['State', 'District', 'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Yield', 'Rainfall', 'Temperature', 'Season_Encoded', 'State_Encoded', 'Crop_Encoded', 'Log_Yield', 'Log_Area', 'Log_Production']


In [ ]:
from google.colab import files
files.download("../data/processed/crop_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>